# PyTorch extension: cohort-aware RAI2 regression

This notebook follows up on the public-cohort analysis with a small model for RAI2 expression. I implemented it **after the thesis** while learning PyTorch; it was not part of the submitted thesis.

## Why a separate analysis was created

The first version trained a TCGA model to classify TMPRSS2–ERG status and applied it unchanged to SU2C. The ERG-excluding primary classifier showed only weak discrimination on the TCGA internal test set (ROC-AUC 0.632; balanced accuracy 0.560) and in SU2C (ROC-AUC 0.608). At the fixed 0.5 threshold it classified **all 181 SU2C samples as negative**, giving sensitivity 0, specificity 1 and balanced accuracy 0.5.

The ERG-inclusive positive control separated samples very well by rank (internal ROC-AUC 0.983; SU2C ROC-AUC 0.905), but it also classified every SU2C sample as negative at 0.5. Its external Brier score of 0.530 showed that the absolute probabilities were badly calibrated. A good ROC-AUC can coexist with failed threshold classification because ROC-AUC uses only the ordering of scores.

This failure is biologically and technically plausible. TCGA and SU2C differ in tumour stage, patient composition, laboratory methods and expression processing. The result therefore demonstrates cross-cohort domain shift rather than a transferable clinical classifier.

## Revised question and study design

The revised analysis no longer tries to infer a fusion event that is already measured directly. It asks: **how much out-of-fold information about continuous RAI2 expression is provided by TMPRSS2–ERG status and CTBP1/CTBP2 within each cohort?**

- **Target:** corrected $\log_2$(RAI2 expression + 1)
- **Primary predictors:** fusion status, CTBP1 and CTBP2
- **Comparisons:** CTBP1/CTBP2 without fusion status, and fusion status alone
- **Cohort strategy:** TCGA and SU2C are modelled separately and interpreted as independent cohorts
- **Model:** ridge-regularised linear regression implemented with PyTorch
- **Evaluation:** five-fold outer cross-validation with four-fold inner selection of ridge strength

I used a linear model because there are at most three predictors. For this data structure it is easier to interpret, and more appropriate, than a larger neural network.

In [ ]:
from pathlib import Path
import torch

from rai2_regression import (
    COHORT_ORDER, ML_VALUE_COLUMN, MODEL_SPECS, TARGET_GENE,
    load_regression_samples, run_experiments,
)

EXPRESSION_CSV = Path("data/expression_values.csv")
OUTPUT_DIR = Path("results")
print("PyTorch", torch.__version__)
print("Target:", TARGET_GENE)
print("Expression input column:", ML_VALUE_COLUMN)

## Inspect the sample-level data

The tidy expression table generated by `analysis.py` is pivoted to one row per sample. Only complete ERG, RAI2, CTBP1 and CTBP2 records are retained. The model uses corrected `log2_expression`, not the legacy TCGA decimal-stripping value retained solely to reproduce the thesis figure.

In [ ]:
samples = load_regression_samples(EXPRESSION_CSV)
for study_id in COHORT_ORDER:
    cohort = [sample for sample in samples if sample.study_id == study_id]
    positive = sum(sample.fusion_status for sample in cohort)
    print(
        f"{cohort[0].study}: {len(cohort)} samples; "
        f"{positive} fusion-positive and {len(cohort) - positive} comparison"
    )

for model_name, predictors in MODEL_SPECS.items():
    print(f"{model_name}: {', '.join(predictors)}")

## Leakage-aware nested cross-validation

Each cohort is divided into five outer folds stratified by fusion status. One fold is predicted by a model that has never seen it. Within the remaining four folds, a second four-fold loop selects the ridge penalty. Predictor scaling and RAI2 target scaling are fitted only on the corresponding training data. After cycling through all outer folds, every sample has exactly one out-of-fold prediction.

This design avoids choosing model settings on the observations used for final evaluation and is less dependent on one fortunate train/test split.

In [ ]:
metadata = run_experiments(
    EXPRESSION_CSV,
    OUTPUT_DIR,
    seed=42,
    outer_folds=5,
    inner_folds=4,
)
print(f"Outputs written to {OUTPUT_DIR.resolve()}")

## Cross-validated performance

$R^2$ measures the fraction of RAI2 variation explained out of fold; 0 corresponds to predicting the cohort mean and negative values are worse than that baseline. RMSE and MAE are prediction errors in corrected log2-expression units, so lower values are better. Pearson $r$ measures agreement in ranking and linear variation.

In [ ]:
from IPython.display import HTML, display

metric_rows = []
for cohort in metadata["cohorts"].values():
    for model_name, result in cohort["models"].items():
        metrics = result["metrics"]
        metric_rows.append(
            f'''<tr><td>{cohort['study']}</td>
            <td>{model_name.replace('_', ' ')}</td>
            <td>{', '.join(result['features'])}</td><td>{metrics['n']}</td>
            <td>{metrics['r_squared']:.3f}</td><td>{metrics['rmse']:.3f}</td>
            <td>{metrics['mae']:.3f}</td><td>{metrics['pearson_r']:.3f}</td></tr>'''
        )

display(HTML(
    '''<div style="overflow-x:auto"><table style="border-collapse:collapse;white-space:nowrap">
    <thead><tr><th>Cohort</th><th>Model</th><th>Predictors</th><th>n</th>
    <th>R²</th><th>RMSE</th><th>MAE</th><th>Pearson r</th></tr></thead><tbody>'''
    + ''.join(metric_rows)
    + '''</tbody></table></div>'''
))

## Incremental information from fusion status

The central comparison is the change in outer-CV $R^2$ when fusion status is added to the CTBP1/CTBP2 baseline. A positive difference means that fusion status contributes predictive information about RAI2 beyond the two CtBP measurements. This remains an association and must not be interpreted as a causal effect.

In [ ]:
for cohort in metadata["cohorts"].values():
    baseline = cohort["models"]["ctbp_baseline"]["metrics"]["r_squared"]
    primary = cohort["models"]["fusion_ctbp_primary"]["metrics"]["r_squared"]
    fusion_coefficient = (
        cohort["models"]["fusion_ctbp_primary"]
        ["coefficient_summary"]["fusion_status"]["mean"]
    )
    print(f"{cohort['study']}")
    print(f"  Δ outer-CV R² after adding fusion status: {primary - baseline:+.3f}")
    print(f"  Mean standardized fusion coefficient: {fusion_coefficient:+.3f}")

## View the generated figures

The first figure compares observed RAI2 expression with outer-fold predictions from the primary model. The second compares standardized primary-model coefficients. Coefficient points are outer-fold means and lines show their range, making instability across folds visible rather than hiding it.

In [ ]:
from IPython.display import SVG, display

prediction_figure = (OUTPUT_DIR / "rai2_predictions.svg").read_text(encoding="utf-8")
coefficient_figure = (OUTPUT_DIR / "coefficient_comparison.svg").read_text(encoding="utf-8")
display(SVG(data=prediction_figure))
display(SVG(data=coefficient_figure))

## Conclusion and limitations

The earlier classifier was not a robust cross-cohort model. Its external threshold failure is consistent with differences in tumour stage, cohort composition, laboratory protocols and expression processing. The revised model therefore does not claim that TCGA and SU2C share one transferable measurement scale.

Instead, it tests the biologically closer question of whether fusion status adds information about RAI2 expression **within each cohort**. Similar directions in both independently analysed cohorts would support a reproducible association; different effect sizes or predictive performance would indicate context dependence. Low $R^2$ values would mean that fusion status and CTBP1/CTBP2 explain only a small part of sample-to-sample RAI2 variation, even when group-level statistical differences exist.

Important limitations remain: the analysis is observational; fusion calls may be incomplete; only a small prespecified gene set is available; correlated predictors complicate coefficient interpretation; and predictive association cannot demonstrate direct regulation. Experimental perturbation data remain necessary for causal conclusions.